# Diabetic Hospital Readmission — Data Cleaning Pipeline

**Dataset:** `fairlearn.datasets.fetch_diabetes_hospital`  
**Goal:** Predict early readmission (`<30` days) — binary classification.  
**Pipeline Steps:**
1. Load & Inspect
2. Fix Target Variable
3. Drop Leakage & High-Missing Columns
4. Replace String `'Missing'` with `NaN`
5. Impute Missing Values
6. Encode Categoricals
7. Final Validation

In [13]:
# ============================================================
# Cell 1 — Imports
# ============================================================
import sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from fairlearn.datasets import fetch_diabetes_hospital

print("Python:", sys.executable)
print(f"pandas {pd.__version__} | numpy {np.__version__}")

Python: c:\Users\Atharv\Downloads\final_pjk\.venv\Scripts\python.exe
pandas 3.0.1 | numpy 2.4.3


In [14]:
# ============================================================
# Cell 2 — Load & Inspect
# ============================================================
def load_dataset() -> pd.DataFrame:
    data = fetch_diabetes_hospital(as_frame=True)
    
    df = data.data.copy()
    
    # Safety check
    assert 'readmitted' in df.columns, "readmitted column missing!"
    
    return df


df = load_dataset()

print("Shape          :", df.shape)
print("Columns        :", df.columns.tolist())
print("\nDtypes:\n", df.dtypes.value_counts())
print("\nFirst 3 rows:")
df.head(3)

Shape          : (101766, 24)
Columns        : ['race', 'gender', 'age', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'primary_diagnosis', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'insulin', 'change', 'diabetesMed', 'medicare', 'medicaid', 'had_emergency', 'had_inpatient_days', 'had_outpatient_days', 'readmitted', 'readmit_binary']

Dtypes:
 str         13
int64        6
category     5
Name: count, dtype: int64

First 3 rows:


,race,gender,age,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,...,insulin,change,diabetesMed,medicare,medicaid,had_emergency,had_inpatient_days,had_outpatient_days,readmitted,readmit_binary
0,Caucasian,Female,'30 years or younger',Other,Referral,1,Other,41,0,1,...,No,No,No,False,False,False,False,False,NO,0
1,Caucasian,Female,'30 years or younger','Discharged to Home',Emergency,3,Missing,59,0,18,...,Up,Ch,Yes,False,False,False,False,False,>30,1
2,AfricanAmerican,Female,'30 years or younger','Discharged to Home',Emergency,2,Missing,11,5,13,...,No,No,Yes,False,False,False,True,True,NO,0


In [15]:
# ============================================================
# Cell 3 — Fix Target Variable
# ============================================================
# BUG FIX: data.target from fairlearn is already numeric (readmit_binary),
# NOT the string 'readmitted' column. Using data.target directly with
# x == '<30' would ALWAYS return 0 — silently breaking the target.
#
# Correct approach: build the binary label from the 'readmitted' string
# column that is already present in data.data.
#   <30  → 1  (early readmission — high risk)
#   >30 / NO → 0

def make_target(df: pd.DataFrame, col: str = 'readmitted') -> pd.Series:
    """
    Creates a binary target: 1 if readmitted within 30 days, else 0.
    """
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found. Available: {df.columns.tolist()}")

    return (df[col] == '<30').astype(int)


# Create target
df['target'] = make_target(df)

# 🚨 CRITICAL FIX — prevent target leakage
df.drop(columns=['readmitted'], inplace=True)

print("Target distribution:")
print(df['target'].value_counts())
print(f"\nPositive rate: {df['target'].mean():.2%}")

# Sanity check — must have BOTH classes present
assert df['target'].nunique() == 2, \
    "ERROR: target column only has one class — check original values!"

assert df['target'].isin([0, 1]).all(), \
    "ERROR: unexpected values in target column."

print("\n✅ Target check passed.")

Target distribution:
target
0    90409
1    11357
Name: count, dtype: int64

Positive rate: 11.16%

✅ Target check passed.


In [16]:
# ============================================================
# Cell 4 — Drop Leakage & High-Missing Columns
# ============================================================
# Columns to drop and why:
#   encounter_id   — unique row identifier, no predictive value
#   patient_nbr    — unique patient identifier, no predictive value
#   weight         — ~97% missing in original dataset
#   payer_code     — ~40% missing and low predictive relevance
#   medical_specialty — ~49% missing (confirmed in EDA)
#   readmitted     — raw source of our target (data leakage)
#   readmit_binary — direct derivation of target (data leakage)

COLS_TO_DROP = [
    'encounter_id',
    'patient_nbr',
    'weight',
    'payer_code',
    'medical_specialty',
    'readmitted',       # leakage — source of target
    'readmit_binary',   # leakage — direct derivation of target
]

before = df.shape[1]
df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns], inplace=True)
after = df.shape[1]

print(f"Dropped {before - after} columns. New shape: {df.shape}")
print("Remaining columns:", df.columns.tolist())
df = df.drop_duplicates()


# 🚨 CRITICAL ADDITION — remove duplicates
before_rows = df.shape[0]
df = df.drop_duplicates()
after_rows = df.shape[0]

print(f"Removed {before_rows - after_rows} duplicate rows. New shape: {df.shape}")

Dropped 2 columns. New shape: (101766, 22)
Remaining columns: ['race', 'gender', 'age', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'primary_diagnosis', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'insulin', 'change', 'diabetesMed', 'medicare', 'medicaid', 'had_emergency', 'had_inpatient_days', 'had_outpatient_days', 'target']
Removed 0 duplicate rows. New shape: (101740, 22)


In [17]:
# ============================================================
# Cell 5 — Replace String 'Missing' with NaN
# ============================================================
# The dataset encodes absent values as the string 'Missing'.
# We unify them with proper NaN so pandas imputation works correctly.

def replace_string_missing(df: pd.DataFrame, sentinel: str = 'Missing') -> pd.DataFrame:
    """
    Replaces a sentinel string (e.g. 'Missing') with np.nan across all columns.
    Uses vectorized pd.DataFrame.replace — no Python-level loops.
    """
    return df.replace(sentinel, np.nan)


df = replace_string_missing(df)

# Report missing counts
missing = df.isnull().sum()
missing_nonzero = missing[missing > 0].sort_values(ascending=False)
if missing_nonzero.empty:
    print("No missing values found after replacement.")
else:
    print("Columns with NaN after replacement:")
    print(missing_nonzero)

No missing values found after replacement.


In [18]:
# ============================================================
# Cell 6 — Impute Missing Values
# ============================================================
# Strategy:
#   Categorical / string columns → fill with 'Unknown'
#   Numeric columns              → fill with column median
#
# Pandas 2+/4+ compliance:
#   - Use include='str' instead of deprecated include='object'
#   - Avoid chained inplace assignment (copy-on-write safe)

def impute_missing(df: pd.DataFrame) -> pd.DataFrame:
    """
    Imputes missing values in-place (copy-on-write compatible).

    - String / categorical columns: fills NaN with 'Unknown'.
    - Numeric columns: fills NaN with the column median.

    Returns the imputed DataFrame.
    """
    df = df.copy()  # avoid mutating caller's reference

    # --- Categorical columns ---
    try:
        # pandas 3+: use 'str' dtype selector
        str_cols = df.select_dtypes(include='str').columns
    except Exception:
        # fallback for pandas < 3
        str_cols = df.select_dtypes(include='object').columns

    df[str_cols] = df[str_cols].fillna('Unknown')

    # --- Numeric columns (excluding the target) ---
    num_cols = df.select_dtypes(include='number').columns.difference(['target'])
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    return df


df = impute_missing(df)

remaining_nulls = df.isnull().sum().sum()
print(f"Missing values after imputation: {remaining_nulls}")
assert remaining_nulls == 0, "ERROR: nulls still present after imputation!"
print("✅ Imputation check passed.")

Missing values after imputation: 0
✅ Imputation check passed.


In [19]:
def encode_categoricals(
    df: pd.DataFrame,
    exclude_cols: list = None
) -> tuple[pd.DataFrame, OrdinalEncoder, list]:

    exclude_cols = exclude_cols or []
    df = df.copy()

    # ✅ FIXED: robust categorical column detection
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    cat_cols = [col for col in cat_cols if col not in exclude_cols]

    if not cat_cols:
        print("No categorical columns to encode.")
        return df, None, []

    # ⚠️ Optional but strong safety check
    assert df[cat_cols].isnull().sum().sum() == 0, \
        "Categorical columns contain NaNs — imputation must be done before encoding!"

    encoder = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1,
        dtype=np.int32
    )

    df[cat_cols] = encoder.fit_transform(df[cat_cols])

    return df, encoder, cat_cols


df, cat_encoder, encoded_cols = encode_categoricals(df, exclude_cols=['target'])

print(f"Encoded {len(encoded_cols)} categorical column(s): {encoded_cols}")
print("\nDtypes after encoding:")
print(df.dtypes.value_counts())

C:\Users\Atharv\AppData\Local\Temp\ipykernel_18592\1378364229.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object', 'category']).columns


Encoded 16 categorical column(s): ['race', 'gender', 'age', 'discharge_disposition_id', 'admission_source_id', 'primary_diagnosis', 'max_glu_serum', 'A1Cresult', 'insulin', 'change', 'diabetesMed', 'medicare', 'medicaid', 'had_emergency', 'had_inpatient_days', 'had_outpatient_days']

Dtypes after encoding:
int32    16
int64     6
Name: count, dtype: int64


In [20]:
# ============================================================
# Cell 7.5 — Feature Engineering (FIXED VERSION)
# ============================================================

def feature_engineering(df):
    
    df = df.copy()
    
    # ✅ Check columns before using them
    cols = df.columns
    
    # 1. Medication intensity (SAFE)
    if 'num_medications' in cols and 'time_in_hospital' in cols:
        df['medication_ratio'] = df['num_medications'] / (df['time_in_hospital'] + 1)
    
    # 2. Lab intensity (SAFE)
    if 'num_lab_procedures' in cols and 'time_in_hospital' in cols:
        df['lab_ratio'] = df['num_lab_procedures'] / (df['time_in_hospital'] + 1)
    
    # 3. Procedure intensity
    if 'num_procedures' in cols and 'time_in_hospital' in cols:
        df['procedure_ratio'] = df['num_procedures'] / (df['time_in_hospital'] + 1)
    
    # 4. Total hospital activity (NEW)
    if all(col in cols for col in ['num_lab_procedures', 'num_procedures', 'num_medications']):
        df['total_activity'] = (
            df['num_lab_procedures'] +
            df['num_procedures'] +
            df['num_medications']
        )
    
    return df


df = feature_engineering(df)

print("Feature engineering applied.")
print("New shape:", df.shape)

Feature engineering applied.
New shape: (101740, 26)


In [21]:
# ============================================================
# Cell 8 — Final Validation
# ============================================================

def validate_pipeline(df: pd.DataFrame) -> None:
    """
    Runs a suite of assertions to confirm the cleaned DataFrame
    is ready for model training.
    """
    print("=" * 50)
    print("PIPELINE VALIDATION REPORT")
    print("=" * 50)

    # 1. Shape
    print(f"\n📐 Shape          : {df.shape}")
    assert df.shape[0] > 0, "DataFrame is empty!"

    # 2. No missing values
    nulls = df.isnull().sum().sum()
    print(f"❓ Missing values  : {nulls}")
    assert nulls == 0, f"Found {nulls} missing values!"

    # 3. Target integrity
    assert 'target' in df.columns, "'target' column missing!"
    assert df['target'].isin([0, 1]).all(), "Target has values outside {0, 1}!"
    dist = df['target'].value_counts()
    pos_rate = df['target'].mean()
    print(f"🎯 Target classes  : {dist.to_dict()}")
    print(f"   Positive rate   : {pos_rate:.2%}")
    assert df['target'].nunique() == 2, "Target must have exactly 2 classes!"

    # 4. No object/string columns remaining
    try:
        obj_cols = df.select_dtypes(include='str').columns.tolist()
    except Exception:
        obj_cols = df.select_dtypes(include='object').columns.tolist()
    print(f"🔤 Remaining str cols: {obj_cols}")
    assert len(obj_cols) == 0, f"Unenoded string columns still present: {obj_cols}"

    # 5. No leakage columns
    leakage_guard = ['readmitted', 'readmit_binary']
    found_leakage = [c for c in leakage_guard if c in df.columns]
    print(f"🚨 Leakage columns : {found_leakage}")
    assert len(found_leakage) == 0, f"Leakage columns still present: {found_leakage}"

    # 6. Duplicates
    dups = df.duplicated().sum()
    print(f"👥 Duplicate rows  : {dups}")

    print("\n" + "=" * 50)
    print("✅ ALL CHECKS PASSED — dataset ready for modelling.")
    print("=" * 50)


validate_pipeline(df)
# Check duplicates
dup_count = df.duplicated().sum()
print(f"🧬 Duplicate rows : {dup_count}")
assert dup_count == 0, "Duplicates still exist!"
print("\nSample (first 5 rows):")
df.head()

PIPELINE VALIDATION REPORT

📐 Shape          : (101740, 26)
❓ Missing values  : 0
🎯 Target classes  : {0: 90383, 1: 11357}
   Positive rate   : 11.16%
🔤 Remaining str cols: []
🚨 Leakage columns : []
👥 Duplicate rows  : 0

✅ ALL CHECKS PASSED — dataset ready for modelling.
🧬 Duplicate rows : 0

Sample (first 5 rows):


,race,gender,age,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,primary_diagnosis,...,medicare,medicaid,had_emergency,had_inpatient_days,had_outpatient_days,target,medication_ratio,lab_ratio,procedure_ratio,total_activity
0,2,0,0,1,2,1,41,0,1,3,...,0,0,0,0,0,0,0.500000,20.500000,0.000000,42
1,2,0,0,0,0,3,59,0,18,4,...,0,0,0,0,0,0,4.500000,14.750000,0.000000,77
2,0,0,0,0,0,2,11,5,13,4,...,0,0,0,1,1,0,4.333333,3.666667,1.666667,29
3,2,1,1,0,0,2,44,1,16,4,...,0,0,0,0,0,0,5.333333,14.666667,0.333333,61
4,2,1,1,0,0,1,51,0,8,4,...,0,0,0,0,0,0,4.000000,25.500000,0.000000,59


In [24]:
df.to_csv("cleaned_data.csv", index=False)